# Scraping data from Riot API & Processing the data to dataframes

In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
import csv
import numpy as np

load_dotenv()

api_key = os.getenv("RIOT_API_KEY")
print(api_key)

# Get PUUID from gameName and tagLine
def get_puuid(gameName=None, tagLine=None, region=None, api_key=None):
    if gameName is None or tagLine is None:
        raise ValueError("gameName and tagLine must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()['puuid']
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get gameName and tagLine from PUUID
def get_name_and_tag(puuid=None, region=None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-puuid/{puuid}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data['gameName'], data['tagLine']
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None, None

# Get ladder data for a specific region and league
def get_ladder(region=None, league=None, api_key=None):
    if region is None:
        raise ValueError("region must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/league/v4/{league}leagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match history for a specific PUUID
def get_match_history(puuid=None, region=None, count = None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/by-puuid/{puuid}/ids?count={count}&api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match data for a specific match ID
def get_match_data_from_id(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match timeline for a specific match ID
def get_match_timeline(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}/timeline?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Process match data into a DataFrame
def process_match_to_df(match_data):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    match_id = metadata.get('matchId')
    all_participants_puuids = metadata.get('participants', [])
    
    # Stats
    base_info = {
        'matchId': match_id,
        'all_participants': str(all_participants_puuids), 
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion') # Current patch
    }

    # Bans és Objectives
    teams_stats = {}
    for team in info.get('teams', []):
        t_id = team.get('teamId')
        
        # Bans
        team_bans = [ban.get('championId') for ban in team.get('bans', [])]
        
        # Objectives
        objs = team.get('objectives', {})
        obj_data = {
            'team_baron_kills': objs.get('baron', {}).get('kills'),
            'team_dragon_kills': objs.get('dragon', {}).get('kills'),
            'team_horde_kills': objs.get('horde', {}).get('kills'),
            'team_inhibitor_kills': objs.get('inhibitor', {}).get('kills'),
            'team_riftHerald_kills': objs.get('riftHerald', {}).get('kills'),
            'team_tower_kills': objs.get('tower', {}).get('kills')
        }
        
        teams_stats[t_id] = {
            'bans': team_bans,
            **obj_data
        }

    # ParticipantDto
    rows = []
    for p in info.get('participants', []):
        p_stats = {
            'assists': p.get('assists'),
            'champLevel': p.get('champLevel'),
            'championId': p.get('championId'),
            'deaths': p.get('deaths'),
            'detectorWardsPlaced': p.get('detectorWardsPlaced'),
            'damageDealtToObjectives': p.get('damageDealtToObjectives'),
            'damageSelfMitigated': p.get('damageSelfMitigated'),
            'firstBloodKill': p.get('firstBloodKill'),
            'firstTowerKill': p.get('firstTowerKill'),
            'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
            'gameEndedInSurrender': p.get('gameEndedInSurrender'),
            'goldEarned': p.get('goldEarned'),
            'item0': p.get('item0'),
            'item1': p.get('item1'),
            'item2': p.get('item2'),
            'item3': p.get('item3'),
            'item4': p.get('item4'),
            'item5': p.get('item5'),
            'item6': p.get('item6'),
            'kills': p.get('kills'),
            'lane': p.get('lane'),
            'neutralMinionsKilled': p.get('neutralMinionsKilled'),
            'objectivesStolen': p.get('objectivesStolen'),
            'participantId': p.get('participantId'),
            'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
            'totalDamageTaken': p.get('totalDamageTaken'),
            'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
            'visionScore': p.get('visionScore'),
            'win': p.get('win'),
            'teamId': p.get('teamId'),
            'riotIdGameName': p.get('riotIdGameName')
        }
        
        combined_row = {**base_info, **p_stats, **teams_stats.get(p.get('teamId'), {})}
        rows.append(combined_row)

    return pd.DataFrame(rows)

# Get player-specific stats from match data
def get_player_stats_from_match(match_data, target_puuid):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    p = next((player for player in info.get('participants', []) if player.get('puuid') == target_puuid), None)
    
    if not p:
        print(f"A játékos ({target_puuid}) nem szerepelt ebben a meccsben.")
        return None

    player_stats = {
        # Match - MetadataDto
        'matchId': metadata.get('matchId'),
        'participants': metadata.get('participants'),
        
        # Match - InfoDto
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion'),
        
        # ParticipantDto
        'assists': p.get('assists'),
        'champLevel': p.get('champLevel'),
        'championId': p.get('championId'),
        'deaths': p.get('deaths'),
        'detectorWardsPlaced': p.get('detectorWardsPlaced'),
        'damageDealtToObjectives': p.get('damageDealtToObjectives'),
        'damageSelfMitigated': p.get('damageSelfMitigated'),
        'firstBloodKill': p.get('firstBloodKill'),
        'firstTowerKill': p.get('firstTowerKill'),
        'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
        'gameEndedInSurrender': p.get('gameEndedInSurrender'),
        'goldEarned': p.get('goldEarned'),
        'item0': p.get('item0'), 'item1': p.get('item1'), 'item2': p.get('item2'),
        'item3': p.get('item3'), 'item4': p.get('item4'), 'item5': p.get('item5'),
        'item6': p.get('item6'),
        'kills': p.get('kills'),
        'lane': p.get('lane'),
        'neutralMinionsKilled': p.get('neutralMinionsKilled'),
        'objectivesStolen': p.get('objectivesStolen'),
        'participantId': p.get('participantId'),
        'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
        'totalDamageTaken': p.get('totalDamageTaken'),
        'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
        'visionScore': p.get('visionScore'),
        'win': p.get('win'),
        'riotIdGameName': p.get('riotIdGameName')
    }

    # Bans & Objectives
    player_team_id = p.get('teamId')
    team_data = next((t for t in info.get('teams', []) if t.get('teamId') == player_team_id), {})
    
    # BanDto
    player_stats['bans'] = [ban.get('championId') for ban in team_data.get('bans', [])]
    
    # ObjectivesDto
    objs = team_data.get('objectives', {})
    player_stats['team_baron_kills'] = objs.get('baron', {}).get('kills')
    player_stats['team_dragon_kills'] = objs.get('dragon', {}).get('kills')
    player_stats['team_horde_kills'] = objs.get('horde', {}).get('kills')
    player_stats['team_inhibitor_kills'] = objs.get('inhibitor', {}).get('kills')
    player_stats['team_riftHerald_kills'] = objs.get('riftHerald', {}).get('kills')
    player_stats['team_tower_kills'] = objs.get('tower', {}).get('kills')

    return player_stats

# Process timeline data into a DataFrame
def process_timeline_to_frames(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    rows = []
    
    # Frame-ek feldolgozása
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        timestamp = frame['timestamp']
        
        # Current data
        for p_id, p_info in frame['participantFrames'].items():
            row = {
                'matchId': match_id,
                'minute': frame_idx,
                'timestamp': timestamp,
                'participantId': int(p_id),
                'totalGold': p_info.get('totalGold'),
                'currentGold': p_info.get('currentGold'),
                'xp': p_info.get('xp'),
                'level': p_info.get('level'),
                'minionsKilled': p_info.get('minionsKilled'),
                'jungleMinionsKilled': p_info.get('jungleMinionsKilled'),
                'positionX': p_info.get('position', {}).get('x'),
                'positionY': p_info.get('position', {}).get('y')
            }
            rows.append(row)
            
    return pd.DataFrame(rows)

# Process timeline events into a DataFrame
def process_timeline_events(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    events_list = []
    
    for frame in timeline_data['info']['frames']:
        for event in frame.get('events', []):
            if event['type'] in ['CHAMPION_KILL', 'ELITE_ monster_kill', 'BUILDING_KILL']: 
                event_row = {
                    'matchId': match_id,
                    'timestamp': event.get('timestamp'),
                    'type': event.get('type'),
                    'killerId': event.get('killerId'),
                    'victimId': event.get('victimId'),
                    'monsterType': event.get('monsterType'),
                    'teamId': event.get('teamId'),
                    'positionX': event.get('position', {}).get('x'),
                    'positionY': event.get('position', {}).get('y')
                }
                events_list.append(event_row)
                
    return pd.DataFrame(events_list)

RGAPI-9361ca3b-74c1-42b4-ae95-806bbb6f7119


In [2]:
player = get_puuid(gameName="Peyz", tagLine="KR11", region="asia", api_key=api_key)
player

'SlQZ2zlvU0VGYNvISzXkK8v6bTF_gFr44fnHfMPZH8QaLZYmMo_XjpJ28nMAlKVBWjtIt3xZUSCA-Q'

In [ ]:
# Regions & Leagues
#regions = ["kr", "euw1", "na1", "br1", "eun1", "jp1", "la1", "la2", "me1", "oc1", "ru", "sg2", "tr1", "tw2", "vn2"]
regions = ["na1"]
leagues = ["challenger", "grandmaster"]

all_ladders = []

# Limit counter
request_count = 0

for region in regions:
    for league in leagues:
        
        # RATE LIMIT
        request_count += 1
        
        # If we hit 100 requests, wait for 125 seconds to respect API limits
        if request_count % 100 == 0:
            print(f"\nLimit reached: {request_count} requests. Waiting for 125 seconds to respect API limits...")
            time.sleep(125)
        # If we hit 20 requests, wait for 2 seconds to avoid hitting short-term limits
        elif request_count % 20 == 0:
            print(f"\n[-] Limit reached: {request_count} requests. Waiting for 2 seconds...")
            time.sleep(2)
        else:
            time.sleep(0.05)

        print(f"Query: {region.upper()} - {league} (Request #{request_count})")
        data = get_ladder(region=region, league=league, api_key=api_key)
        
        if data and 'entries' in data:
            df = pd.DataFrame(data['entries'])
            
            # New columns for region and league
            df['region'] = region
            df['league'] = league
            
            # Append to the list of DataFrames
            all_ladders.append(df)

# Combine all ladders into a single DataFrame
if all_ladders:
    na_challenger_grandmaster_df = pd.concat(all_ladders, ignore_index=True)
    print(f"\nSuccess! {len(na_challenger_grandmaster_df)} players' data loaded.")
else:
    print("\nFailed to retrieve data.")

na_challenger_grandmaster_df.head()

In [ ]:
na_challenger_grandmaster_df.to_csv("na_challenger_grandmaster.csv", index=False)

In [ ]:
try:
    # Read the player data from the CSV file
    players_df = pd.read_csv("na_challenger_grandmaster.csv")
    
except FileNotFoundError:
    print("Error: na_challenger_grandmaster.csv not found.")
    exit()

unique_match_ids = set() # No duplicates
request_count = 0

total_players = len(players_df)
print(f"Total players to process: {total_players}.")

# Iterate through each player and get their match history
for index, row in enumerate(players_df.itertuples(), 1):
    puuid = row.puuid
    # Routing region is based on the server region (na1, kr, euw1, etc.)
    server_region = row.region 
    
    # Region mapping for Match-V5 API 
    if server_region in ['kr', 'jp1', 'tw2', 'vn2']:
        routing_region = "asia"
    elif server_region in ['euw1', 'eun1', 'tr1', 'ru', 'me1']:
        routing_region = "europe"
    elif server_region in ['na1', 'br1', 'la1', 'la2']:
        routing_region = "americas"
    else:
        routing_region = "sea"

    print(f"[{index}/{total_players}] Querying {puuid[:8]}... ({routing_region})")
    
    # Rate Limit Handling
    request_count += 1
    if request_count % 100 == 0:
        print(f"\n[!] 100 requests reached. Waiting for 125 seconds to respect API limits")
        time.sleep(125)
    elif request_count % 20 == 0:
        time.sleep(2)
    else:
        time.sleep(0.05)
    
    # Maximum 100 matches per request
    matches = get_match_history(puuid=puuid, region=routing_region, count=100, api_key=api_key)
    
    if matches:
        # Update the set of unique match IDs with the new matches
        unique_match_ids.update(matches)
    elif matches is None:
         # Request failed (Riot Server error)
         time.sleep(5)

matches_df = pd.DataFrame(list(unique_match_ids), columns=['matchId'])

# Save the unique match IDs to a CSV file
matches_df.to_csv("na_matches.csv", index=False)
print("The unique match IDs have been saved to 'na_matches.csv'.")

matches_df.head()

In [ ]:
# Match IDs
#matches_df = pd.read_csv("kr_matches.csv")
match_ids = matches_df['matchId'].tolist()


print(f"Processing {len(match_ids)} matches...")

# Counters and storage
request_count = 0
batch_size = 100 # Write to file every 100 matches to manage memory
all_frames = []
all_events = []
batch_index = 1

# Iterate through each match ID and get the timeline data
for index, match_id in enumerate(match_ids, 1):
    print(f"[{index}/{len(match_ids)}] Getting timeline for: {match_id}...")
    
    # Rate Limit Handling
    request_count += 1
    if request_count % 100 == 0:
        print("\n[!] 1100 requests reached. Waiting for 125 seconds to respect API limits")
        time.sleep(125)
    elif request_count % 20 == 0:
        time.sleep(2)
    else:
        time.sleep(0.05)
    
    timeline_data = get_match_timeline(match_id=match_id, region="asia", api_key=api_key)
    
    if timeline_data and 'info' in timeline_data:
        try:
            frames_df = process_timeline_to_frames(timeline_data)
            events_df = process_timeline_events(timeline_data)
            
            all_frames.append(frames_df)
            all_events.append(events_df)
        except Exception as e:
            print(f"Error processing match {match_id}: {e}")
    else:
        print(f"Failed to fetch or empty response for match {match_id}")

    # Save in batches to manage memory
    if index % batch_size == 0 or index == len(match_ids):
        print(f"\n--- Saving in progress (Batch {batch_index}) ---")
        
        if all_frames:
            batch_frames_df = pd.concat(all_frames, ignore_index=True)
            batch_events_df = pd.concat(all_events, ignore_index=True)
            
            # Append
            write_header = (batch_index == 1) # Only write header for the first batch
            
            batch_frames_df.to_csv("timeline_frames_kr.csv", mode='a', index=False, header=write_header)
            batch_events_df.to_csv("timeline_events_kr.csv", mode='a', index=False, header=write_header)
            
            print(f"Batch {batch_index} saved! (+{len(batch_frames_df)} frame rows, +{len(batch_events_df)} event rows)")
            
            # Clear the lists for the next batch
            all_frames = []
            all_events = []
            batch_index += 1
            
print("\n--- ALL MATCHES PROCESSED AND SAVED! ---")

In [3]:
def process_timeline_to_frames_full(timeline_data):
    if not timeline_data or 'info' not in timeline_data:
        return None
        
    match_id = timeline_data['metadata']['matchId']
    rows = []
    
    # Process each frame in the timeline
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        timestamp = frame['timestamp']
        
        # Current data for each participant
        #for p_id, p_info in frame['participantFrames'].items():

        p_frames = frame.get('participantFrames')
        if not p_frames: 
            continue
            
        for p_id, p_info in p_frames.items():
            
            # Basic Stats
            row = {
                'matchId': match_id,
                'minute': frame_idx,
                'timestamp': timestamp,
                'participantId': int(p_id),
                'totalGold': p_info.get('totalGold'),
                'currentGold': p_info.get('currentGold'),
                'xp': p_info.get('xp'),
                'level': p_info.get('level'),
                'minionsKilled': p_info.get('minionsKilled'),
                'jungleMinionsKilled': p_info.get('jungleMinionsKilled'),
                'timeEnemySpentControlled': p_info.get('timeEnemySpentControlled')
            }
            
            # Position Data
            pos = p_info.get('position', {})
            row['positionX'] = pos.get('x')
            row['positionY'] = pos.get('y')
            
            # Champion Stats
            champ_stats = p_info.get('championStats', {})
            for stat_name, stat_value in champ_stats.items():
                row[f'stat_{stat_name}'] = stat_value
                
            # Damage Stats
            damage_stats = p_info.get('damageStats', {})
            for dmg_name, dmg_value in damage_stats.items():
                row[f'dmg_{dmg_name}'] = dmg_value
                
            rows.append(row)
            
    return pd.DataFrame(rows)

In [4]:
def process_timeline_events_full(timeline_data):
    if not timeline_data or 'info' not in timeline_data:
        return None

    match_id = timeline_data['metadata']['matchId']
    events_list = []
    
    # Iterate through each frame and its events
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        
        # Iterate through all events in the current frame
        for event in frame.get('events', []):
            
            # Basic event info
            event_row = {
                'matchId': match_id,
                'minute': frame_idx, # Frame index as minute
            }
            
            # All event fields
            for key, value in event.items():
                
                # Position
                if key == 'position' and isinstance(value, dict):
                    event_row['positionX'] = value.get('x')
                    event_row['positionY'] = value.get('y')
                    
                # Stringify lists
                elif isinstance(value, list):
                    event_row[key] = str(value)
                    
                # Nested dictionaries
                elif isinstance(value, dict):
                    for sub_key, sub_val in value.items():
                        event_row[f"{key}_{sub_key}"] = sub_val
                        
                # Other fields
                else:
                    event_row[key] = value
                    
            events_list.append(event_row)
            
    return pd.DataFrame(events_list)

In [ ]:
# Settings
INPUT_CSV = "Data/Matches/kr_matches.csv" # Match IDs
OUTPUT_FOLDER = "Data/Timelines/Raw_Batches" # Output folder for raw batches

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Read match IDs
matches_df = pd.read_csv(INPUT_CSV)
col_name = 'matchId' if 'matchId' in matches_df.columns else matches_df.columns[0]
match_ids = matches_df[col_name].dropna().unique().tolist()

# Batching
request_count = 0
batch_size = 500 # Save every 500 matches to manage memory
all_frames = []
all_events = []
batch_index = 1

print(f"Downloading {len(match_ids)} matches.")

for index, match_id in enumerate(match_ids, 1):
    print(f"[{index}/{len(match_ids)}] Timeline lekérése: {match_id}...")
    
    # Rate Limit Handling
    request_count += 1
    if request_count % 100 == 0:
        time.sleep(125) # 100 requests / 2 minute limit
    elif request_count % 20 == 0:
        time.sleep(2)   # 20 requests / second limit (safety buffer)
    else:
        time.sleep(0.05)
        
    # API Call - Get timeline data for the match
    timeline_data = get_match_timeline(match_id=match_id, region="asia", api_key=api_key)
    
    if timeline_data and 'info' in timeline_data:
        frames_df = process_timeline_to_frames_full(timeline_data)
        events_df = process_timeline_events_full(timeline_data)
        
        if frames_df is not None: all_frames.append(frames_df)
        if events_df is not None: all_events.append(events_df)

    # Saving in batches to manage memory and avoid data loss
    if index % batch_size == 0 or index == len(match_ids):
        print(f"\n Saving Batch {batch_index}")
        
        if all_frames:
            batch_frames_df = pd.concat(all_frames, ignore_index=True)
            frames_filename = os.path.join(OUTPUT_FOLDER, f"frames_batch_{batch_index}.csv")
            # No append
            batch_frames_df.to_csv(frames_filename, index=False)
            
        if all_events:
            batch_events_df = pd.concat(all_events, ignore_index=True)
            events_filename = os.path.join(OUTPUT_FOLDER, f"events_batch_{batch_index}.csv")
            # No append
            batch_events_df.to_csv(events_filename, index=False)
            
        print(f"Batch {batch_index} saved!")
        
        # Clear the lists for the next batch
        all_frames = []
        all_events = []
        batch_index += 1

print("\n Done!")

[1/41070] Timeline lekérése: KR_8104946260...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[2/41070] Timeline lekérése: KR_8079420256...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[3/41070] Timeline lekérése: KR_8100723627...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[4/41070] Timeline lekérése: KR_8075146353...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[5/41070] Timeline lekérése: KR_8117825010...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[6/41070] Timeline lekérése: KR_8123001991...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[7/41070] Timeline lekérése: KR_8126152049...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[8/41070] Timeline lekérése: KR_8115452323...
Error: 401 - {"status":{"message":"Unknown apikey","status_code":401}}
[9/41070] Timeline lekérése: KR_8096792206...
Error: 401 - {"sta

KeyboardInterrupt: 

In [ ]:
def split_csv(input_file, output_folder, file_prefix, matches_per_file=1000):
    print(f"Splitting: {input_file}")
    
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    with open(input_file, 'r', encoding='utf-8') as f_in:
        reader = csv.reader(f_in)
        header = next(reader) # Header row
        
        # Finding matchId index in the header
        try:
            match_id_idx = header.index('matchId')
        except ValueError:
            print("Error: 'matchId' column not found in the header!")
            return

        current_file_idx = 1
        current_match_count = 0
        current_match_id = None
        
        # Open the first output file
        out_filename = os.path.join(output_folder, f"{file_prefix}_part_{current_file_idx}.csv")
        f_out = open(out_filename, 'w', encoding='utf-8', newline='')
        writer = csv.writer(f_out)
        writer.writerow(header)
        
        print(f"Writing to: {out_filename}...")

        # Iterate through each row and split based on matchId
        for row in reader:
            # Skip empty rows
            if not row:
                continue
                
            row_match_id = row[match_id_idx]
            
            # If we encounter a new matchId, we check if we need to switch files
            if row_match_id != current_match_id:
                current_match_id = row_match_id
                current_match_count += 1
                
                # Switch to a new file if we've reached the limit of matches per file
                if current_match_count > matches_per_file:
                    f_out.close() # Close the current file
                    
                    current_file_idx += 1
                    current_match_count = 1 # Reset the match count for the new file
                    
                    # Open the new file
                    out_filename = os.path.join(output_folder, f"{file_prefix}_part_{current_file_idx}.csv")
                    f_out = open(out_filename, 'w', encoding='utf-8', newline='')
                    writer = csv.writer(f_out)
                    writer.writerow(header) # Header row written to the new file
                    
                    print(f"Reached the limit of {matches_per_file} matches. Switching to a new file: {out_filename}")
            
            # Write the current row to the currently open file
            writer.writerow(row)
            
        # Close the last file
        f_out.close()
        
    print(f"\nSplitting complete! A total of {current_file_idx} files were created in the '{output_folder}' folder.")

In [ ]:
split_csv("Data/Timelines/timeline_frames_full_kr.csv", "Data/Timelines/Frames_Chunked", "frames_kr", 1000)

## Making compact DF

In [13]:
import ast

# Read the first batch
frames_df = pd.read_csv("Data/Timelines/Raw_Batches/frames_batch_5.csv", low_memory=False)
events_df = pd.read_csv("Data/Timelines/Raw_Batches/events_batch_5.csv", low_memory=False)

# Extract target variable (Win) from GAME_END event before filtering
win_df = events_df[events_df['type'] == 'GAME_END'][['matchId', 'winningTeam']].copy()
if 'winningTeam' in win_df.columns:
    win_df['winningTeam'] = pd.to_numeric(win_df['winningTeam'], errors='coerce')
    win_df = win_df.dropna(subset=['winningTeam']).drop_duplicates('matchId')
    win_df['target_win_t100'] = (win_df['winningTeam'] == 100).astype(int)
    win_df = win_df[['matchId', 'target_win_t100']]
else:
    win_df = pd.DataFrame(columns=['matchId', 'target_win_t100'])

# Convert ID columns to numeric
id_cols = ['killerId', 'participantId', 'creatorId']

for col in id_cols:
    if col in events_df.columns:
        events_df[col] = pd.to_numeric(events_df[col], errors='coerce')

# Processing
keys = ['matchId', 'minute', 'timestamp', 'participantId']
value_cols = [c for c in frames_df.columns if c not in keys]

# Pivot to wide format
wide_frames = frames_df.pivot_table(
    index=['matchId', 'minute', 'timestamp'],
    columns='participantId',
    values=value_cols
).reset_index()

# Flatten the multi-level columns
wide_frames.columns = [
    f"{col[0]}_p{int(col[1])}" if col[1] != '' else col[0]
    for col in wide_frames.columns
]

# Create team-level aggregates and differentials
for feature in value_cols:
    try:
        wide_frames[f'{feature}_t100'] = wide_frames[[f'{feature}_p{i}' for i in range(1, 6)]].sum(axis=1)
        wide_frames[f'{feature}_t200'] = wide_frames[[f'{feature}_p{i}' for i in range(6, 11)]].sum(axis=1)
        wide_frames[f'{feature}_diff'] = wide_frames[f'{feature}_t100'] - wide_frames[f'{feature}_t200']
    except:
        pass

# Process events

# Helper function to parse damage strings
def parse_damage_list(dmg_str):
    if pd.isna(dmg_str) or type(dmg_str) != str: 
        return 0, 0, 0
    try:
        dmg_list = ast.literal_eval(dmg_str)
        phys = sum(d.get('physicalDamage', 0) for d in dmg_list)
        magic = sum(d.get('magicDamage', 0) for d in dmg_list)
        true_d = sum(d.get('trueDamage', 0) for d in dmg_list)
        return phys, magic, true_d
    except:
        return 0, 0, 0

# Extract pre-death damage for CHAMPION_KILL events
kill_mask = events_df['type'] == 'CHAMPION_KILL'
if 'victimDamageReceived' in events_df.columns:
    parsed_dmg = events_df.loc[kill_mask, 'victimDamageReceived'].apply(parse_damage_list)
    events_df.loc[kill_mask, 'kill_phys_dmg_received'] = parsed_dmg.apply(lambda x: x[0])
    events_df.loc[kill_mask, 'kill_magic_dmg_received'] = parsed_dmg.apply(lambda x: x[1])
    events_df.loc[kill_mask, 'kill_true_dmg_received'] = parsed_dmg.apply(lambda x: x[2])

# Only keep relevant event types
events_df = events_df[
    events_df['type'].isin([
        'CHAMPION_KILL',
        'ELITE_MONSTER_KILL',
        'BUILDING_KILL',
        'WARD_PLACED',
        'WARD_KILL',
        'ITEM_PURCHASED',
        'ITEM_SOLD',
        'ITEM_DESTROYED',
        'SKILL_LEVEL_UP',
        'LEVEL_UP',
        'TURRET_PLATE_DESTROYED'
    ])
].copy()

# Minutes
events_df['minute'] = (events_df['timestamp'] // 60000).astype(int)

# Assigning teams based on killerId and event type
events_df['actor_id'] = events_df[['participantId', 'killerId', 'creatorId']].bfill(axis=1).iloc[:, 0]
events_df['team'] = 'neutral'

# Assign teams to all valid actors
events_df.loc[events_df['actor_id'].between(1, 5), 'team'] = 't100'
events_df.loc[events_df['actor_id'].between(6, 10), 'team'] = 't200'

# Only valid teams
events_df = events_df[events_df['team'].isin(['t100','t200'])]

# Aggregate event counts by match, minute, team, and type
event_counts = (
    events_df
    .groupby(['matchId', 'minute', 'team', 'type'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Pivot to wide format for event counts
event_pivot_counts = event_counts.pivot_table(
    index=['matchId', 'minute'],
    columns='team',
    aggfunc='sum',
    fill_value=0
)
event_pivot_counts.columns = [f"{col[0]}_{col[1]}" for col in event_pivot_counts.columns]
event_pivot_counts = event_pivot_counts.reset_index()

# Aggregate damage sums for CHAMPION_KILL
dmg_cols = ['kill_phys_dmg_received', 'kill_magic_dmg_received', 'kill_true_dmg_received']
if all(c in events_df.columns for c in dmg_cols):
    event_sums = (
        events_df[events_df['type'] == 'CHAMPION_KILL']
        .groupby(['matchId', 'minute', 'team'])[dmg_cols]
        .sum()
        .unstack(fill_value=0)
    )
    event_sums.columns = [f"{col[0]}_{col[1]}" for col in event_sums.columns]
    event_sums = event_sums.reset_index()
    
    # Combine event counts and damage sums
    event_pivot = pd.merge(event_pivot_counts, event_sums, on=['matchId', 'minute'], how='outer').fillna(0)
else:
    event_pivot = event_pivot_counts

# Ensure all minutes are represented for each match
valid_minutes = wide_frames[['matchId', 'minute']].drop_duplicates()

event_pivot = pd.merge(
    valid_minutes,
    event_pivot,
    on=['matchId', 'minute'],
    how='left'
).fillna(0)

# Cumulative sums
event_pivot = event_pivot.sort_values(['matchId', 'minute'])

event_cols = [c for c in event_pivot.columns if c not in ['matchId', 'minute']]

for col in event_cols:
    event_pivot[f'cum_{col}'] = event_pivot.groupby('matchId')[col].cumsum()

# Calculate differences between teams for each event type
event_types = list(set([c.rsplit('_', 1)[0] for c in event_cols]))

for e_type in event_types:
    col_t100 = f'cum_{e_type}_t100'
    col_t200 = f'cum_{e_type}_t200'

    val_t100 = event_pivot.get(col_t100, 0)
    val_t200 = event_pivot.get(col_t200, 0)
    
    event_pivot[f'cum_{e_type}_diff'] = val_t100 - val_t200

# Merge the event data back into the main dataset
ml_dataset = pd.merge(
    wide_frames,
    event_pivot,
    on=['matchId', 'minute'],
    how='left'
)

# Fill NaN values in event-related columns with 0
event_related_cols = [c for c in ml_dataset.columns if 'cum_' in c]
ml_dataset[event_related_cols] = ml_dataset[event_related_cols].fillna(0)

# Add the target variable (win) to the final dataset
ml_dataset = pd.merge(ml_dataset, win_df, on='matchId', how='inner')

# Remove arena matches
if 'totalGold_p11' in ml_dataset.columns:
    arena_matches = ml_dataset[ml_dataset['totalGold_p11'].notna()]['matchId'].unique()
    ml_dataset = ml_dataset[~ml_dataset['matchId'].isin(arena_matches)]

    cols_to_drop = [
        c for c in ml_dataset.columns
        if any(p in c for p in ['_p11','_p12','_p13','_p14','_p15','_p16'])
    ]
    ml_dataset = ml_dataset.drop(columns=cols_to_drop)

# Done
print(f"\nDone! ML dataset: {ml_dataset.shape[0]} rows, {ml_dataset.shape[1]} columns")

C:\Users\hp\AppData\Local\Temp\ipykernel_42868\1194144345.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide_frames[f'{feature}_t200'] = wide_frames[[f'{feature}_p{i}' for i in range(6, 11)]].sum(axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_42868\1194144345.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide_frames[f'{feature}_diff'] = wide_frames[f'{feature}_t100'] - wide_frames[f'{feature}_t200']
C:\Users\hp\AppData\Local\Temp\ipykernel_42868\1194144345.py:44: PerformanceWarning: DataFrame is highly fragmented


Done! ML dataset: 13223 rows, 672 columns


In [14]:
ml_dataset.to_csv("Data/ml_dataset_kr_part5_new.csv", index=False)

In [6]:
# Summary of cumulative columns
cum_cols = [c for c in ml_dataset.columns if 'cum_' in c]
print(f"Total number of columns: {len(ml_dataset.columns)}")
print(f"Number of cumulative columns: {len(cum_cols)}")
print("\nList of created cumulative columns:")
for c in cum_cols:
    print(f" - {c}")

# Testing cumsum
test_match = ml_dataset['matchId'].iloc[-1]


cumsum_test = ml_dataset[ml_dataset['matchId'] == test_match][
    ['matchId', 'minute', 'cum_CHAMPION_KILL_t100', 'cum_CHAMPION_KILL_t200', 'cum_CHAMPION_KILL_diff', 'ITEM_PURCHASED_t100', 'ITEM_PURCHASED_t200', 'cum_ITEM_PURCHASED_diff']
]

# 40 minute of the last match
display(cumsum_test.head(40))

Total number of columns: 672
Number of cumulative columns: 42

List of created cumulative columns:
 - cum_BUILDING_KILL_t100
 - cum_BUILDING_KILL_t200
 - cum_CHAMPION_KILL_t100
 - cum_CHAMPION_KILL_t200
 - cum_ELITE_MONSTER_KILL_t100
 - cum_ELITE_MONSTER_KILL_t200
 - cum_ITEM_DESTROYED_t100
 - cum_ITEM_DESTROYED_t200
 - cum_ITEM_PURCHASED_t100
 - cum_ITEM_PURCHASED_t200
 - cum_ITEM_SOLD_t100
 - cum_ITEM_SOLD_t200
 - cum_LEVEL_UP_t100
 - cum_LEVEL_UP_t200
 - cum_SKILL_LEVEL_UP_t100
 - cum_SKILL_LEVEL_UP_t200
 - cum_TURRET_PLATE_DESTROYED_t100
 - cum_TURRET_PLATE_DESTROYED_t200
 - cum_WARD_KILL_t100
 - cum_WARD_KILL_t200
 - cum_WARD_PLACED_t100
 - cum_WARD_PLACED_t200
 - cum_kill_phys_dmg_received_t100
 - cum_kill_phys_dmg_received_t200
 - cum_kill_magic_dmg_received_t100
 - cum_kill_magic_dmg_received_t200
 - cum_kill_true_dmg_received_t100
 - cum_kill_true_dmg_received_t200
 - cum_ITEM_SOLD_diff
 - cum_WARD_KILL_diff
 - cum_ELITE_MONSTER_KILL_diff
 - cum_ITEM_DESTROYED_diff
 - cum_SKIL

,matchId,minute,cum_CHAMPION_KILL_t100,cum_CHAMPION_KILL_t200,cum_CHAMPION_KILL_diff,ITEM_PURCHASED_t100,ITEM_PURCHASED_t200,cum_ITEM_PURCHASED_diff
13555,KR_8127825809,0,0.0,0.0,0.0,15.0,14.0,1.0
13556,KR_8127825809,1,0.0,0.0,0.0,0.0,1.0,0.0
13557,KR_8127825809,2,0.0,1.0,-1.0,3.0,0.0,3.0
13558,KR_8127825809,3,0.0,1.0,-1.0,3.0,2.0,4.0
13559,KR_8127825809,4,0.0,1.0,-1.0,2.0,2.0,4.0
13560,KR_8127825809,5,2.0,2.0,0.0,7.0,6.0,5.0
13561,KR_8127825809,6,3.0,3.0,0.0,3.0,7.0,1.0
13562,KR_8127825809,7,3.0,3.0,0.0,2.0,0.0,3.0
13563,KR_8127825809,8,4.0,3.0,1.0,11.0,9.0,5.0
13564,KR_8127825809,9,5.0,3.0,2.0,2.0,2.0,5.0


# Merging batches

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Read (First 5 batches)
INPUT_FOLDER = "Data/ML_Ready_Datasets"
sample_batches = []

print("Looking for files...")
for i in range(1, 6):
    file_path = os.path.join(INPUT_FOLDER, f"ml_dataset_kr_part{i}_new.csv")
    
    if os.path.exists(file_path):
        print(f"[OK] Found file: {file_path}")
        df = pd.read_csv(file_path, low_memory=False)
        sample_batches.append(df)
    else:
        print(f"[MISSING] Cannot find file: {file_path}")

# Check if we actually loaded anything
if len(sample_batches) == 0:
    print("\nERROR: No files were loaded! Please check the INPUT_FOLDER path and filenames.")
else:
    # Concatenate into one EDA DataFrame
    eda_df = pd.concat(sample_batches, ignore_index=True)

    print("\n=== 1. DATASET SHAPE ===")
    print(f"Total rows (minute snapshots): {eda_df.shape[0]}")
    print(f"Total columns (features): {eda_df.shape[1]}")
    print(f"Unique matches in sample: {eda_df['matchId'].nunique()}")
    
    print("\nSaving to eda_sample.csv...")
    try:
        eda_df.to_csv("eda_sample.csv", index=False)
        print("[SUCCESS] File saved successfully! You can now open eda_sample.csv")
    except Exception as e:
        print(f"[ERROR] Failed to save CSV: {e}")

Looking for files...
[OK] Found file: Data/ML_Ready_Datasets\ml_dataset_kr_part1_new.csv
[OK] Found file: Data/ML_Ready_Datasets\ml_dataset_kr_part2_new.csv
[OK] Found file: Data/ML_Ready_Datasets\ml_dataset_kr_part3_new.csv
[OK] Found file: Data/ML_Ready_Datasets\ml_dataset_kr_part4_new.csv
[OK] Found file: Data/ML_Ready_Datasets\ml_dataset_kr_part5_new.csv

=== 1. DATASET SHAPE ===
Total rows (minute snapshots): 66785
Total columns (features): 672
Unique matches in sample: 2475

Saving to eda_sample.csv...
[SUCCESS] File saved successfully! You can now open eda_sample.csv
